In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from imutils.perspective import four_point_transform

def resizer(image,width=500):
    # get width and height
    h,w,c=image.shape
    height=int((h/w)*width)
    size=(width,height)
    image=cv2.resize(image,size)
    return image,size


def document_scanner(image):
    
    img_re,size=resizer(img)

    detail=cv2.detailEnhance(img_re,sigma_s=20,sigma_r=0.15)
    gray=cv2.cvtColor(detail,cv2.COLOR_BGR2GRAY)
    blur=cv2.GaussianBlur(gray,(5,5),0)

    #detecting the edges
    edge_image=cv2.Canny(blur,75,200)

    #morphological transform
    kernel=np.ones((5,5),np.uint8)
    dilate=cv2.dilate(edge_image,kernel,iterations=1)
    closing=cv2.morphologyEx(dilate,cv2.MORPH_CLOSE,kernel)

    #find the contours
    contours, hire=cv2.findContours(closing,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    contours= sorted(contours,key=cv2.contourArea,reverse=True)
    for contour in contours:
        peri=cv2.arcLength(contour,True)
        approx=cv2.approxPolyDP(contour,0.02*peri,True)

        if len(approx) ==4:
            four_points= np.squeeze(approx)
            break

    cv2.drawContours(img_re,[four_points],-1,(0,255,0),3)

    #find four points for original image
    multiplier=img.shape[1]/size[0]
    four_points_original=four_points*multiplier
    four_points_original=four_points_original.astype(int)

    wrap_image=four_point_transform(img,four_points_original)

    return wrap_image, four_points_original, img_re, closing

img=cv2.imread('Extracted/frame_0003.jpg')
crop,points, cnt_img, edgeimg=document_scanner(img)
print(points)
cv2.imshow('original image',img)
cv2.imshow('edges image',cnt_img)
cv2.imshow('cropped image',crop)
cv2.waitKey(0)
cv2.destroyAllWindows()

gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
threshold=cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,20)
cv2.imshow('thresholded',threshold)
cv2.waitKey(0)
cv2.destroyAllWindows()

[[295  58]
 [161  61]
 [160  68]
 [293  65]]


In [3]:
import cv2
import numpy as np

def scan_detection(image):
    HEIGHT,WIDTH,c=image.shape
    document_contour=np.array([ [0,0],[WIDTH,0],[WIDTH,HEIGHT],[0,HEIGHT] ])
    gray=cv2.cvtColor(image,cv2.COLOR_BGR2GRAY)
    blur=cv2.GaussianBlur(gray,(5,5),0)
    _,threshold=cv2.threshold(blur,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    contours,_ =cv2.findContours(threshold,cv2.RETR_LIST,cv2.CHAIN_APPROX_SIMPLE)
    contours=sorted(contours,key=cv2.contourArea,reverse=True)

    max_area=0
    for contour in contours:
        area=cv2.contourArea(contour)
        if area > 1000:
            peri=cv2.arcLength(contour,True)
            approx=cv2.approxPolyDP(contour,0.015*peri,True)
            if area > max_area and len(approx)==4:
                document_contour=approx
                max_area=area

    cv2.drawContours(image,[document_contour],-1,(0,255,0),3)

img=cv2.imread('Extracted/frame_0001.jpg')
cv2.imshow('og',img)
cv2.waitKey(0)
cv2.destroyAllWindows()
scan_detection(img)
cv2.imshow('drawn image',img)
cv2.waitKey(0)
cv2.destroyAllWindows()


## For Optical Flow testing

In [2]:
import cv2
import numpy as np
import time

def draw_flow(img, flow, step=16):
    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy= flow[y,x].T

    lines = np.vstack([x, y, x-fx, y-fy]).T.reshape(-1,2,2)
    lines = np.int32(lines + 0.5)

    img_bgr = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.polylines(img_bgr, lines, 0, (0,255,0) )

    for(x1, y1), (x2, y2) in lines:
        cv2.circle(img_bgr, (x1, y1), 1, (0, 255, 0), -1)

    return img_bgr

def draw_hsv(flow):
    h, w = flow.shape[:2]
    fx, fy = flow[:,:,0], flow [:,:,1]

    ang = np.arctan2(fy, fx) + np.pi
    v = np.sqrt(fx*fx +fy*fy)

    hsv = np.zeros( (h, w, 3), np.uint8)
    hsv[...,0] = ang*(180/np.pi/2)
    hsv[...,1] = 255
    hsv[...,2] = np.minimum(v*4, 255)
    bgr =cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    return bgr

cap = cv2.VideoCapture(0)

suc, prev = cap.read()
prevgray = cv2.cvtColor(prev, cv2.COLOR_BGR2GRAY)

while True:
    suc, img = cap.read()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # start time to calculate FPS
    start = time.time()

    flow = cv2.calcOpticalFlowFarneback(prevgray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    prevgray = gray
    
    # End time
    end = time.time()
    # Calculate the FPS for current frame detection
    fps = 1/(end - start)

    print(f"{fps:.2f} FPS")

    cv2.imshow('flow', cv2.flip(draw_flow(gray, flow), 1) )
    cv2.imshow('flow HSV', cv2.flip(draw_hsv(flow), 1) )
    
    key = cv2.waitKey(5)
    if key == ord('q'):
        break


cap.release()
cv2.destroyAllWindows()

15.41 FPS
5.96 FPS
10.42 FPS
15.02 FPS
13.40 FPS
17.10 FPS
7.53 FPS
7.12 FPS
8.32 FPS
7.03 FPS
7.56 FPS
4.52 FPS
4.70 FPS
3.00 FPS
5.17 FPS
4.46 FPS
6.64 FPS
6.84 FPS
6.45 FPS
6.35 FPS
7.25 FPS
6.53 FPS
6.67 FPS
6.62 FPS
7.85 FPS
7.49 FPS
6.17 FPS
6.49 FPS
6.59 FPS
6.54 FPS
4.34 FPS
4.02 FPS
2.80 FPS
4.85 FPS
5.34 FPS
5.48 FPS
5.62 FPS
4.63 FPS
4.70 FPS
4.71 FPS
5.10 FPS
5.07 FPS
5.35 FPS
5.57 FPS
5.77 FPS
3.62 FPS
5.13 FPS
5.75 FPS
7.04 FPS
14.66 FPS
14.51 FPS
15.97 FPS
14.60 FPS
14.66 FPS
14.82 FPS
14.85 FPS
12.52 FPS
14.20 FPS
15.16 FPS
14.79 FPS
9.93 FPS
6.98 FPS
13.72 FPS
15.05 FPS
15.54 FPS
14.71 FPS
17.43 FPS
16.21 FPS
16.87 FPS
17.41 FPS
17.53 FPS
21.10 FPS
16.90 FPS
15.83 FPS
17.71 FPS
18.52 FPS
19.84 FPS
17.57 FPS
15.72 FPS
17.39 FPS
17.69 FPS
14.64 FPS
20.11 FPS
18.12 FPS
17.49 FPS
17.83 FPS
18.40 FPS
17.68 FPS
19.37 FPS
17.96 FPS
17.07 FPS
17.68 FPS
19.29 FPS
18.51 FPS
17.92 FPS
17.63 FPS
17.93 FPS
18.77 FPS
17.66 FPS
16.53 FPS
20.61 FPS
19.44 FPS
19.19 FPS
19.01 FPS
19.61 

In [1]:
import cv2
import os
import numpy as np

# Setup
video_path = "ProjectVideo2.mp4"
output_folder = "Extracted"
os.makedirs(output_folder, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
STABLE_LIMIT = 0.15 # Adjust based on noise
page_count = 0

def get_gray_at_time(seconds):
    """Utility to grab a grayscale frame at a specific timestamp."""
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ret, frame = cap.read()
    if not ret: return None, None
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Downscale for speed
    gray_small = cv2.resize(gray, (320, 180))
    return gray_small, frame

def calculate_motion(gray1, gray2):
    """Calculates optical flow between two frames."""
    if gray1 is None or gray2 is None: return 999.0
    flow = cv2.calcOpticalFlowFarneback(gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    return np.mean(np.abs(flow))

def find_stable_point(temp_sec):
    """
    Recursively checks temp-0.1 and temp+0.1 to find 
    the 'center' of a stable window.
    """
    g_mid, f_mid = get_gray_at_time(temp_sec)
    g_prev, _ = get_gray_at_time(temp_sec - 0.1)
    g_next, _ = get_gray_at_time(temp_sec + 0.1)

    motion_back = calculate_motion(g_prev, g_mid)
    motion_fwd = calculate_motion(g_mid, g_next)

    # Logic: If both sides are stable, we are in the heart of the page
    if motion_back < STABLE_LIMIT and motion_fwd < STABLE_LIMIT:
        return temp_sec, f_mid
    
    # If the back is unstable, the page just arrived; move forward
    if motion_back >= STABLE_LIMIT:
        return find_stable_point(temp_sec + 0.1)
    
    # If the front is unstable, the page is about to turn; move backward
    if motion_fwd >= STABLE_LIMIT:
        return find_stable_point(temp_sec - 0.1)
    
    return None, None

# --- MAIN EXECUTION ---
current_time = 0.1
while True:
    # Phase 1: Linear search for the first/next stable image
    g0, _ = get_gray_at_time(current_time - 0.1)
    g1, f1 = get_gray_at_time(current_time)
    
    if g1 is None: break # End of video

    if calculate_motion(g0, g1) < STABLE_LIMIT:
        # STABLE IMAGE FOUND
        page_count += 1
        cv2.imwrite(f"{output_folder}/page_{page_count}.jpg", f1)
        print(f"Page {page_count} saved at {current_time:.2f}s")

        # Phase 2: The 2-second Jump & Recursive Adjustment
        jump_time = current_time + 2.0
        stable_time, stable_frame = find_stable_point(jump_time)
        
        if stable_time:
            current_time = stable_time
            # Note: The loop will continue and save this stable_frame in the next iteration
        else:
            current_time += 0.1 # Fallback if jump lands in void
    else:
        current_time += 0.1

cap.release()

Page 1 saved at 10.20s


KeyboardInterrupt: 

## for page text structuring

In [11]:
import cv2
import numpy as np
import pytesseract
from PIL import Image
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch

# ────────────────────────────────────────────────
# CONFIG
# ────────────────────────────────────────────────
FONT_SIZE_TITLE      = 72
FONT_SIZE_HEADING    = 36
FONT_SIZE_NORMAL     = 14

POSTER_MODE_MAX_WORDS = 20
POSTER_MODE_MAX_LINES = 6

# Vertical spacing (in points)
BASE_LINE_SPACING    = 1.15     # multiplier × font size
EXTRA_AFTER_TITLE    = 24       # points extra after title lines
EXTRA_AFTER_HEADING  = 12       # points extra after headings

# ────────────────────────────────────────────────

def preprocess_image_cv2(img_path):
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Cannot read image: {img_path}")

    # Your exact preprocessing
    img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    processed = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 20
    )

    # Convert back to PIL for pytesseract (expects PIL or numpy array in some modes)
    processed_pil = Image.fromarray(processed)
    return processed_pil, processed  # return both if needed


# ────────────────────────────────────────────────
# MAIN PROCESSING
# ────────────────────────────────────────────────

image_path = 'Extracted/d41538d7-60e1-4698-9e30-2cd4c278eb3f_frame_0.jpg'          # ← CHANGE THIS
processed_image, _ = preprocess_image_cv2(image_path)

# OCR with better config for documents / posters
data = pytesseract.image_to_data(
    processed_image,
    output_type=pytesseract.Output.DATAFRAME,
    config='--oem 1 --psm 6 -l eng'       # OEM 1 often better after thresholding
)

# Clean & filter
data = data[(data['conf'] > 35) & data['text'].notna() & (data['text'].str.strip() != '')].copy()
data = data.sort_values(['top', 'left'])

if data.empty:
    print("No reliable text detected after preprocessing.")
    exit()

# ────────────────────────────────────────────────
# Global stats for classification
# ────────────────────────────────────────────────

lines_group = data.groupby('line_num').agg({
    'height': 'mean',
    'text': lambda x: ' '.join(x.dropna().tolist())
}).rename(columns={'text': 'full_text'})

lines_group['word_count'] = lines_group['full_text'].str.split().str.len()

global_avg_height   = lines_group['height'].mean()
total_words         = lines_group['word_count'].sum()
total_lines         = len(lines_group)
global_avg_words    = lines_group['word_count'].mean()

is_poster_like = (
    total_words     <= POSTER_MODE_MAX_WORDS or
    total_lines     <= POSTER_MODE_MAX_LINES or
    global_avg_words < 4.0
)

print(f"Image (after ×2 resize): {processed_image.width}×{processed_image.height}")
print(f"Total words: {total_words} | Lines: {total_lines}")
print(f"Avg char height: {global_avg_height:.1f}px | Poster-like: {is_poster_like}")

# ────────────────────────────────────────────────
# Classification
# ────────────────────────────────────────────────

def classify_line(line_height, word_count):
    rel = line_height / global_avg_height if global_avg_height > 0 else 1.0

    if is_poster_like:
        if rel >= 1.35 or word_count <= 6:
            return FONT_SIZE_TITLE, 'title'
        elif rel >= 1.10:
            return FONT_SIZE_HEADING, 'heading'
        else:
            return FONT_SIZE_NORMAL, 'normal'
    else:
        if word_count <= 8 and rel >= 1.70:
            return FONT_SIZE_TITLE, 'title'
        elif word_count <= 14 and rel >= 1.35:
            return FONT_SIZE_HEADING, 'heading'
        elif rel >= 1.15:
            return FONT_SIZE_HEADING - 4, 'subheading'
        else:
            return FONT_SIZE_NORMAL, 'normal'

# ────────────────────────────────────────────────
# Build styled lines + estimate spacing
# ────────────────────────────────────────────────

styled_lines = []

for block_id in sorted(data['block_num'].unique()):
    block = data[data['block_num'] == block_id]
    for par_id in sorted(block['par_num'].unique()):
        par = block[block['par_num'] == par_id]
        par_lines = par.groupby('line_num').agg({
            'left':   'min',
            'top':    'mean',
            'height': 'mean',
            'text':   lambda x: ' '.join(x.dropna().tolist())
        }).rename(columns={'text': 'full_text'})

        par_lines['word_count'] = par_lines['full_text'].str.split().str.len()

        for idx, line in par_lines.iterrows():
            font_size, line_type = classify_line(line['height'], line['word_count'])

            # Clean text per line (your method)
            clean_text = line['full_text'].strip()
            clean_text = clean_text.replace('\u2018', "'").replace('\u2019', "'")
            clean_text = clean_text.replace('\u201c', '"').replace('\u201d', '"')
            clean_text = clean_text.replace('\u2013', '-').replace('\u2014', '-')
            clean_text = clean_text.encode('latin-1', 'ignore').decode('latin-1')

            styled_lines.append({
                'text':      clean_text,
                'x':         line['left'],
                'original_y':line['top'],       # for initial sorting
                'height_px': line['height'],
                'type':      line_type,
                'font_size': font_size
            })

# Sort primarily by original vertical position
styled_lines.sort(key=lambda x: (x['original_y'], x['x']))

# ────────────────────────────────────────────────
# Render PDF with proper vertical flow (no overlap)
# ────────────────────────────────────────────────

c = canvas.Canvas("styled_output.pdf", pagesize=letter)
page_w, page_h = letter

scale_x = page_w / processed_image.width
scale_y = page_h / processed_image.height   # still used for horizontal + rough vertical guide

font_map = {
    'title':      'Helvetica-Bold',
    'heading':    'Helvetica-Bold',
    'subheading': 'Helvetica-BoldOblique',
    'normal':     'Helvetica'
}

current_y = page_h - 1.2 * inch   # start from top, leave margin

for i, line in enumerate(styled_lines):
    font_name = font_map.get(line['type'], 'Helvetica')
    c.setFont(font_name, line['font_size'])

    x = line['x'] * scale_x
    text_width = c.stringWidth(line['text'], font_name, line['font_size'])

    # If text would go off right edge — you can add wrapping or just clip/scale later
    if x + text_width > page_w - 0.5*inch:
        x = page_w - text_width - 0.5*inch   # simple right-align fallback

    c.drawString(x, current_y, line['text'])

    print(f"{line['type'].upper():10} {line['font_size']:3}pt   {line['text']}")

    # Advance y position (downward = smaller y in PDF coords)
    line_spacing = line['font_size'] * BASE_LINE_SPACING

    if line['type'] == 'title':
        line_spacing += EXTRA_AFTER_TITLE
    elif line['type'] in ('heading', 'subheading'):
        line_spacing += EXTRA_AFTER_HEADING

    current_y -= line_spacing

    # Optional: extra paragraph spacing after last line of paragraph
    # (you can detect paragraph change using block/par id comparison)

c.save()

print("\nPDF saved → styled_output.pdf")

Image (after ×2 resize): 956×1700
Total words: 20 | Lines: 14
Avg char height: 18.9px | Poster-like: True
TITLE       72pt   POTN
TITLE       72pt   ee - SRE :
TITLE       72pt   - . SL
TITLE       72pt   i ve
TITLE       72pt   '
TITLE       72pt   |
TITLE       72pt   ne
TITLE       72pt   STANDING
TITLE       72pt   od
TITLE       72pt   :
TITLE       72pt   ~
TITLE       72pt   Te
TITLE       72pt   ae
TITLE       72pt   NS

PDF saved → styled_output.pdf


## For Model Selecting Pages

In [6]:
import os
import cv2
import torch
import numpy as np
from transnetv2_pytorch import TransNetV2

def extract_pages_final_fix(video_path, out_dir='Model Prediction'):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    
    # 1. Load Model
    model = TransNetV2()
    print(f"Deep Scanning Video: {video_path}...")

    # 2. Detect Scenes 
    # Since you have FFmpeg, we use the standard method
    # Lowered threshold to 0.3 to be more sensitive to page turns
    scenes = model.detect_scenes(video_path, threshold=0.3)
    
    print(f"Model returned data type: {type(scenes)}")
    print(f"Raw scene data: {scenes}")

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    saved_count = 0
    
    # 3. Flexible Extraction Loop
    for i, scene in enumerate(scenes):
        # The raw data shows these exact keys: 'start_frame' and 'end_frame'
        try:
            start_f = int(scene['start_frame'])
            end_f = int(scene['end_frame'])
        except Exception as e:
            print(f"Parsing error on scene {i}: {e}")
            continue

        duration = end_f - start_f
        
        # Filter: Ignore scenes shorter than 1.0 second (prevents glitch frames)
        if duration < fps:
            print(f"Skipping short scene {i} ({duration} frames)")
            continue

        # Target the middle of the scene for the best shot
        mid_f = start_f + (duration // 2)
        
        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_f)
        ret, frame = cap.read()
        
        if ret:
            # Sharpness check
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            score = cv2.Laplacian(gray, cv2.CV_64F).var()
            
            # Lowered sharpness threshold to 40 for more reliable extraction
            if score > 40: 
                save_path = os.path.join(out_dir, f"page_{saved_count:03d}.jpg")
                cv2.imwrite(save_path, frame)
                print(f"SUCCESS: Saved Page {saved_count} (Frame {mid_f}, Sharpness: {score:.1f})")
                saved_count += 1
            else:
                print(f"REJECTED: Frame {mid_f} is too blurry ({score:.1f})")

    cap.release()
    print(f"\n--- Extraction Complete ---")
    print(f"Total Unique Pages Found: {saved_count}")

if __name__ == "__main__":
    extract_pages_final_fix('ProjectVideo2.mp4')

Deep Scanning Video: ProjectVideo2.mp4...
Model returned data type: <class 'list'>
Raw scene data: [{'shot_id': 1, 'start_frame': 0, 'end_frame': 30, 'probability': 0.3609192967414856, 'start_time': '0.000', 'end_time': '1.001'}, {'shot_id': 2, 'start_frame': 33, 'end_frame': 138, 'probability': 0.3419208824634552, 'start_time': '1.101', 'end_time': '4.605'}, {'shot_id': 3, 'start_frame': 139, 'end_frame': 558, 'probability': 0.3015696406364441, 'start_time': '4.638', 'end_time': '18.619'}, {'shot_id': 4, 'start_frame': 559, 'end_frame': 566, 'probability': 0.33497121930122375, 'start_time': '18.652', 'end_time': '18.886'}, {'shot_id': 5, 'start_frame': 567, 'end_frame': 768, 'probability': 0.342693954706192, 'start_time': '18.919', 'end_time': '25.626'}, {'shot_id': 6, 'start_frame': 769, 'end_frame': 774, 'probability': 0.30374714732170105, 'start_time': '25.659', 'end_time': '25.826'}, {'shot_id': 7, 'start_frame': 775, 'end_frame': 977, 'probability': 0.3138178586959839, 'start_tim

In [4]:
"""
Fast Smart Frame Selector for Book Video OCR
=============================================
Designed for speed WITHOUT sacrificing frame quality.

What was slow before (removed):
  ✗ Optical flow (Farneback) — extremely CPU heavy, O(W×H) per frame pair
  ✗ SSIM pass that seeks back through video — causes random disk seeks
  ✗ Two-pass video reading — video was read twice

What replaces it (fast equivalents):
  ✓ Frame difference (absolute pixel diff on tiny thumbnail) — replaces optical flow
    → ~200x faster, catches the same motion signal
  ✓ Single linear pass through video — no seeking, no re-reads
  ✓ All analysis done on a small thumbnail (320×180) — not full resolution
  ✓ pHash computed on thumbnail — instant dedup at save time
  ✓ Variance-based sharpness (on thumbnail) — fast, good enough for filtering

Algorithm (single pass):
  Step 1  Read frame → downsample to thumbnail
  Step 2  Skip if frame diff vs previous is HIGH  → motion / page turn
  Step 3  Skip if frame diff vs previous is LOW   → duplicate / same page still
  Step 4  Skip if sharpness (variance) is LOW     → blurry
  Step 5  Skip if exposure is bad                 → too dark / too bright
  Step 6  Cooldown gate: enforce minimum gap between saved frames
          → prevents bursts of near-identical frames from same page
  Step 7  pHash dedup at save time               → final guard against duplicates

Result: one clean representative frame per page, fast.
"""

import cv2
import numpy as np
import os
import time
import logging
from pathlib import Path
from typing import Optional

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  tweak these if results aren't right for your video
# ══════════════════════════════════════════════════════════════════════════════

VIDEO_PATH = "ProjectVideo2.mp4"
OUTPUT_DIR = "Model Prediction"

# ── Sampling ──────────────────────────────────────────────────────────────────
# Analyse every Nth frame. 5 = good speed/quality balance for 25-30fps video.
# For 60fps video, raise to 10. For slow 10fps scans, lower to 2.
SAMPLE_EVERY = 5

# ── Motion / duplicate detection (frame difference) ──────────────────────────
# Mean absolute pixel difference between consecutive thumbnails (0-255 scale).
#
# MOTION_HIGH: diff ABOVE this → frame is mid-turn or camera moving → SKIP
#   Too low = rejects good frames. Too high = keeps blurry turn frames.
#   Raise if too many good frames are being skipped.
MOTION_HIGH = 25

# MOTION_LOW: diff BELOW this → frame looks identical to previous → SKIP
#   This is the main dedup mechanism — same page sitting still = low diff.
#   Raise if you're still getting duplicates. Lower if pages are being missed.
MOTION_LOW = 3

# ── Sharpness ─────────────────────────────────────────────────────────────────
# Pixel variance of the thumbnail grayscale. Blurry frames have low variance.
# Raise if blurry frames are slipping through. Lower if too many frames rejected.
SHARPNESS_MIN = 300

# ── Exposure ──────────────────────────────────────────────────────────────────
# Fraction of pixels allowed in extreme dark (<30) or bright (>225) zones.
MAX_DARK_FRACTION   = 0.20
MAX_BRIGHT_FRACTION = 0.20

# ── Cooldown gate ─────────────────────────────────────────────────────────────
# Minimum number of VIDEO FRAMES that must pass between two saved frames.
# At 25fps, 75 = at least 3 seconds between saves. Prevents same-page bursts.
# Raise for fast page-flipping videos. Lower for slow deliberate scans.
COOLDOWN_FRAMES = 75

# ── pHash deduplication ───────────────────────────────────────────────────────
# Hamming distance below this = same page, skip.
# 12 = tolerant (catches slight crop/lighting changes of same page).
# Lower to 8 for stricter dedup. Raise to 15 if different pages look similar.
PHASH_DEDUP_DIST = 12

# ── Thumbnail size ────────────────────────────────────────────────────────────
# All analysis runs on this size. Smaller = faster. Do not go below 160×90.
THUMB_W = 320
THUMB_H = 180

# ══════════════════════════════════════════════════════════════════════════════


# ── Fast signal functions (all operate on small thumbnails) ───────────────────

def frame_diff(prev: np.ndarray, curr: np.ndarray) -> float:
    """Mean absolute difference between two grayscale thumbnails.
    Near-zero = same scene. Very high = motion/transition."""
    return float(np.mean(np.abs(curr.astype(np.int16) - prev.astype(np.int16))))


def sharpness(gray: np.ndarray) -> float:
    """Pixel variance. Blurry frames cluster near 0, sharp text frames are high."""
    return float(gray.var())


def exposure_ok(gray: np.ndarray) -> bool:
    """True if the frame is neither too dark nor washed out."""
    total       = gray.size
    dark_frac   = np.sum(gray < 30)  / total
    bright_frac = np.sum(gray > 225) / total
    return dark_frac < MAX_DARK_FRACTION and bright_frac < MAX_BRIGHT_FRACTION


def compute_phash(gray: np.ndarray, size: int = 8) -> int:
    """Perceptual hash (pHash). Returns a 64-bit int.
    Two visually similar frames → small Hamming distance."""
    small   = cv2.resize(gray, (size * 4, size * 4))
    dct     = cv2.dct(small.astype(np.float32))
    dct_low = dct[:size, :size]
    med     = np.median(dct_low)
    bits    = (dct_low > med).flatten()
    v = 0
    for b in bits:
        v = (v << 1) | int(b)
    return int(v)


def phash_dist(h1: int, h2: int) -> int:
    return bin(h1 ^ h2).count("1")


# ── Main selector ─────────────────────────────────────────────────────────────

def run():
    if not os.path.exists(VIDEO_PATH):
        raise FileNotFoundError(
            f"\nCould not find '{VIDEO_PATH}'.\n"
            f"Place ProjectVideo2.mp4 in the same folder as this script."
        )

    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open: {VIDEO_PATH}")

    fps        = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total      = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_s = total / fps

    log.info(f"Video   : {VIDEO_PATH}")
    log.info(f"Frames  : {total}  |  FPS: {fps:.1f}  |  Duration: {duration_s/60:.1f} min")
    log.info(f"Output  : {out_dir}/")
    log.info(f"Sampling every {SAMPLE_EVERY} frames")
    log.info("─" * 55)

    # ── State ─────────────────────────────────────────────────────────────────
    prev_thumb:  Optional[np.ndarray] = None
    saved_hashes = []          # pHashes of every saved frame (dedup list)
    saved_count  = 0
    last_saved_idx = -COOLDOWN_FRAMES  # allow saving from frame 0

    t_start = time.time()

    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # ── Skip non-sampled frames (fast) ────────────────────────────────────
        if idx % SAMPLE_EVERY != 0:
            idx += 1
            continue

        # ── Downsample once — all analysis on thumbnail ───────────────────────
        thumb      = cv2.resize(frame, (THUMB_W, THUMB_H))
        thumb_gray = cv2.cvtColor(thumb, cv2.COLOR_BGR2GRAY)

        # ── Gate 1: motion / duplicate via frame diff ─────────────────────────
        if prev_thumb is not None:
            diff = frame_diff(prev_thumb, thumb_gray)

            if diff > MOTION_HIGH:
                # Frame is mid-page-turn or camera is moving — skip
                prev_thumb = thumb_gray
                idx += 1
                continue

            if diff < MOTION_LOW:
                # Identical to previous frame — skip (main dedup path)
                idx += 1
                continue
        prev_thumb = thumb_gray

        # ── Gate 2: sharpness ─────────────────────────────────────────────────
        if sharpness(thumb_gray) < SHARPNESS_MIN:
            idx += 1
            continue

        # ── Gate 3: exposure ──────────────────────────────────────────────────
        if not exposure_ok(thumb_gray):
            idx += 1
            continue

        # ── Gate 4: cooldown (prevent same-page burst) ────────────────────────
        if (idx - last_saved_idx) < COOLDOWN_FRAMES:
            idx += 1
            continue

        # ── Gate 5: pHash dedup (final safety net) ────────────────────────────
        ph = compute_phash(thumb_gray)
        if any(phash_dist(ph, h) < PHASH_DEDUP_DIST for h in saved_hashes):
            idx += 1
            continue

        # ── All gates passed → save the FULL-RESOLUTION frame ─────────────────
        ts    = idx / fps
        fname = f"page_{saved_count+1:04d}_f{idx}_t{ts:.1f}s.jpg"
        path  = str(out_dir / fname)
        cv2.imwrite(path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

        saved_hashes.append(ph)
        last_saved_idx = idx
        saved_count   += 1

        if saved_count % 10 == 0 or saved_count <= 5:
            elapsed = time.time() - t_start
            pct     = idx / max(total, 1) * 100
            log.info(f"  [{pct:5.1f}%]  Saved page {saved_count:4d}  → {fname}  ({elapsed:.0f}s elapsed)")

        idx += 1

    cap.release()

    elapsed = time.time() - t_start
    log.info("─" * 55)
    log.info(f"Done in {elapsed:.1f}s  ({elapsed/60:.1f} min)")
    log.info(f"Saved {saved_count} frames to '{out_dir}/'")

    if saved_count == 0:
        log.warning("No frames were saved! Try adjusting these settings:")
        log.warning("  → Lower SHARPNESS_MIN  (currently %d)", SHARPNESS_MIN)
        log.warning("  → Raise  MOTION_HIGH   (currently %d)", MOTION_HIGH)
        log.warning("  → Lower  MOTION_LOW    (currently %d)", MOTION_LOW)
        log.warning("  → Lower  COOLDOWN_FRAMES (currently %d)", COOLDOWN_FRAMES)


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run()

21:43:40  INFO      Video   : ProjectVideo2.mp4
21:43:40  INFO      Frames  : 2709  |  FPS: 30.0  |  Duration: 1.5 min
21:43:40  INFO      Output  : Model Prediction/
21:43:40  INFO      Sampling every 5 frames
21:43:40  INFO      ───────────────────────────────────────────────────────
21:43:40  INFO        [  0.0%]  Saved page    1  → page_0001_f0_t0.0s.jpg  (0s elapsed)
21:43:40  INFO        [  4.1%]  Saved page    2  → page_0002_f110_t3.7s.jpg  (0s elapsed)
21:43:40  INFO        [  6.8%]  Saved page    3  → page_0003_f185_t6.2s.jpg  (0s elapsed)
21:43:40  INFO        [  9.6%]  Saved page    4  → page_0004_f260_t8.7s.jpg  (0s elapsed)
21:43:41  INFO        [ 12.4%]  Saved page    5  → page_0005_f335_t11.2s.jpg  (0s elapsed)
21:43:41  INFO        [ 26.4%]  Saved page   10  → page_0010_f715_t23.8s.jpg  (1s elapsed)
21:43:42  INFO        [ 54.4%]  Saved page   20  → page_0020_f1475_t49.2s.jpg  (2s elapsed)
21:43:43  INFO        [ 83.8%]  Saved page   30  → page_0030_f2270_t75.7s.jpg  (3